In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import json
import os
import chromadb
from chromadb.utils import embedding_functions
from chromadb.api.models.Collection import Collection
from dotenv import load_dotenv




In [3]:
load_dotenv()

True

In [4]:
chroma_client = chromadb.Client()

In [5]:
collection = chroma_client.create_collection(
    name="games"
)

In [11]:
# Normalize function 

def normalize_game(game, file_id):
    return {
        "id": file_id,
        "name": game.get("Name", "Unknown"),
        "platform": game.get("Platform", "Unknown"),
        "genre": game.get("Genre", "Unknown"),
        "publisher": game.get("Publisher", "Unknown"),
        "description": game.get("Description", "Unknown"),
        "release_date": str(game.get("YearOfRelease", "Unknown"))
    }


In [12]:
games_folder="./games"
games_data=[]

for file_name in os.listdir(games_folder):
    if file_name.endswith(".json"):
        file_path=os.path.join(games_folder,file_name)

        with open(file_path, "r" , encoding="utf-8") as f:
            data =json.load(f)

            file_id=file_name.replace(".json", "")
            games_data.append(normalize_game(data,file_id))
          

print(f"Loaded {len(games_data)} records ")           

Loaded 15 records 


In [15]:
# Insert into vector DB
for game in games_data:
    text = f"""
    Name: {game['name']}
    Genre: {game['genre']}
    Platform: {game['platform']}
    Publisher: {game['publisher']}
    Release Year: {game['release_date']}
    Description: {game['description']}
    """

    collection.add(
        documents=[text],
        ids=[game["id"]],
        metadatas=[game]
    )

print("Data Stored in ChromaDB")


/home/student/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 95.7MiB/s]


Data Stored in ChromaDB


In [19]:
# Semantic search 

def search_games(query,top_k=2):
    return collection.query(
        query_texts=[query],
        n_results=top_k
    )

 

In [20]:
## Test 

#search_games("Tell me about Grand Theft Auto: San Andreas")   

search_games("Tell me about Nintendo Switch")  

{'ids': [['012', '008']],
 'embeddings': None,
 'documents': [['\n    Name: Mario Kart 8 Deluxe\n    Genre: Racing\n    Platform: Nintendo Switch\n    Publisher: Nintendo\n    Release Year: 2017\n    Description: An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.\n    ',
   '\n    Name: Super Mario World\n    Genre: Platformer\n    Platform: Super Nintendo Entertainment System (SNES)\n    Publisher: Nintendo\n    Release Year: 1990\n    Description: A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.\n    ']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'publisher': 'Nintendo',
    'name': 'Mario Kart 8 Deluxe',
    'description': 'An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.',
    'genre': 'Racing',
    'release_date': '2017',
    'platform': 'Nintendo Switch',
